In [11]:
!pip install lxml


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
from lxml import etree
import re

# =========================
# PARAMÈTRES
# =========================
ARK = "bpt6k6568782t"
NOM_FICHIER_XML = "Piles_AbregeViePeintres"

START_F = 12 # mettre un num en moins que la page "vue" à laquelle on commence

BASE_URL = f"https://gallica.bnf.fr/iiif/ark:/12148/{ARK}/f{{}}/full/full/0/native.jpg"

NSMAP = {
    None: "http://www.tei-c.org/ns/1.0",
    "xi": "http://www.w3.org/2001/XInclude"
}

# =========================
# PARSER (IMPORTANT)
# =========================
parser = etree.XMLParser(
    remove_blank_text=True,
    strip_cdata=False
)

tree = etree.parse(f"../../corpus/Peinture/{NOM_FICHIER_XML}.xml", parser)
root = tree.getroot()

# =========================
# EXTRACTION fXXX
# =========================
def extract_f_number(facs_value):
    match = re.search(r'/f(\d+)/', facs_value)
    if match:
        return int(match.group(1))
    return None

# =========================
# TRAITEMENT DES <pb>
# =========================
current_f = START_F
stop = False

for pb in root.xpath('//tei:pb', namespaces={'tei': NSMAP[None]}):
    facs = pb.get('facs')

    if facs == "stop":
        pb.attrib.pop('facs')  # Supprime l'attribut facs="stop"
        stop = True
        continue

    if stop:
        continue

    if facs:
        extracted = extract_f_number(facs)
        if extracted:
            current_f = extracted
    else:
        current_f += 1
        pb.set('facs', BASE_URL.format(current_f))

# =========================
# FIX xi:include (éviter ns1)
# =========================
for el in root.xpath('//*[local-name()="include"]'):
    el.tag = "{http://www.w3.org/2001/XInclude}include"

# =========================
# ÉCRITURE SANS ESPACES EN TROP
# =========================
tree.write(
    "output.xml",
    encoding="UTF-8",
    xml_declaration=True,
    pretty_print=True
)

print("✔ XML propre, namespaces conservés, pas de ns1")

✔ XML propre, namespaces conservés, pas de ns1


GALLICA avec prise en compte de <gap> pour les pages coupées

In [53]:
from lxml import etree
import re

# =========================
# PARAMÈTRES
# =========================
ARK = "bpt6k85631r"
NOM_FICHIER_XML = "Martin_ArchitectureAlberti"

START_F = 0  # mettre un num en moins que la page "vue" à laquelle on commence

BASE_URL = f"https://gallica.bnf.fr/iiif/ark:/12148/{ARK}/f{{}}/full/full/0/native.jpg"

NSMAP = {
    None: "http://www.tei-c.org/ns/1.0",
    "xi": "http://www.w3.org/2001/XInclude"
}
TEI_NS = NSMAP[None]

# =========================
# PARSER (IMPORTANT)
# =========================
parser = etree.XMLParser(
    remove_blank_text=True,
    strip_cdata=False
)

tree = etree.parse(f"../../corpus/Architecture/{NOM_FICHIER_XML}.xml", parser)
root = tree.getroot()

# =========================
# EXTRACTION fXXX
# =========================
def extract_f_number(facs_value):
    match = re.search(r'/f(\d+)/', facs_value)
    if match:
        return int(match.group(1))
    return None

# =========================
# TRAITEMENT DES <pb> ET DES <gap>
# =========================
# On parcourt pb ET gap ensemble, dans l'ordre du document (lxml conserve
# l'ordre du document avec l'opérateur "|"), pour que les <gap quantity="N">
# décalent bien le compteur "current_f" avant le prochain <pb>.
current_f = START_F
stop = False
after_gap = False  # True juste après un <gap> traité, pour forcer le recalcul du pb suivant

elements = root.xpath('//tei:pb | //tei:gap', namespaces={'tei': TEI_NS})

for el in elements:
    tag = etree.QName(el).localname

    # ---- Cas <gap> : décale le compteur de folios sans poser de facs ----
    if tag == 'gap':
        if stop:
            continue

        quantity = el.get('quantity')
        if quantity and quantity.isdigit():
            current_f += int(quantity)
            after_gap = True  # le prochain <pb> devra être recalculé, même s'il a déjà un facs
        else:
            print(f"  [!] <gap> sans @quantity exploitable (attrs={dict(el.attrib)}), ignoré")
        continue

    # ---- Cas <pb> : logique d'origine, avec priorité au <gap> qui précède ----
    facs = el.get('facs')

    if facs == "stop":
        el.attrib.pop('facs')  # Supprime l'attribut facs="stop"
        stop = True
        after_gap = False
        continue

    if stop:
        continue

    if after_gap:
        # Un <gap> vient de décaler le compteur de "quantity" folios sautés.
        # Ce <pb> reste lui-même une page normale : on lui applique le même +1
        # qu'à n'importe quel pb sans facs, et on écrase tout facs déjà présent
        # (forcément calculé sans tenir compte de ce gap).
        ancienne_valeur = facs
        current_f += 1
        el.set('facs', BASE_URL.format(current_f))
        if ancienne_valeur and ancienne_valeur != el.get('facs'):
            print(f"  [i] facs corrigé après <gap> : {ancienne_valeur} → {el.get('facs')}")
        after_gap = False
    elif facs:
        extracted = extract_f_number(facs)
        if extracted:
            current_f = extracted
    else:
        current_f += 1
        el.set('facs', BASE_URL.format(current_f))

# =========================
# FIX xi:include (éviter ns1)
# =========================
for el in root.xpath('//*[local-name()="include"]'):
    el.tag = "{http://www.w3.org/2001/XInclude}include"

# =========================
# ÉCRITURE SANS ESPACES EN TROP
# =========================
tree.write(
    "output.xml",
    encoding="UTF-8",
    xml_declaration=True,
    pretty_print=True
)

print("✔ XML propre, namespaces conservés, pas de ns1, <gap quantity> pris en compte")

✔ XML propre, namespaces conservés, pas de ns1, <gap quantity> pris en compte


In [ ]:
from lxml import etree
import re
from pathlib import Path

# =========================
# PARAMÈTRES
# =========================
ARK = "bd6t57781345"
START_F = 0
START_ARCHIVE = 0

DOCUMENT = "Bassi_DispareriArchitettura.xml"
TYPE_TEXTE = "Architecture"
DOCUMENT_PATH = Path("../../corpus") / TYPE_TEXTE / DOCUMENT

BASE_ARCHIVE_URL = "https://iiif.archive.org/image/iiif/3/gri_pitturexxxxx00albe%2Fgri_pitturexxxxx00albe_jp2.zip%2Fgri_pitturexxxxx00albe_jp2%2Fgri_pitturexxxxx00albe_{:04d}.jp2/full/max/0/default.jpg"

NAMESPACES = {'tei': 'http://www.tei-c.org/ns/1.0', 'xi': 'http://www.w3.org/2001/XInclude'}

# =========================
# PARSER & TRAITEMENT
# =========================
def extract_f_number(facs_value):
    if not facs_value: return None
    match = re.search(r'/f(\d+)', facs_value)
    return int(match.group(1)) if match else None

parser = etree.XMLParser(remove_blank_text=True)
tree = etree.parse(str(DOCUMENT_PATH), parser)
root = tree.getroot()

# Initialisation
current_f = START_F
offset = START_ARCHIVE - START_F

# Correction de la boucle : on suit l'ordre du document
for i, pb in enumerate(root.xpath('//tei:pb', namespaces=NAMESPACES)):
    facs = pb.get('facs')
    
    # Si c'est la toute première page, on reste sur START_F
    # Sinon, on analyse pour voir s'il faut sauter à un nouveau numéro ou juste faire +1
    if i > 0:
        extracted = extract_f_number(facs)
        if extracted:
            current_f = extracted
        else:
            current_f += 1

    # Calcul de l'index Archive
    archive_idx = current_f + offset
    
    # MISE À JOUR SYSTÉMATIQUE
    pb.set('facs', BASE_ARCHIVE_URL.format(archive_idx))
    
    # Petit debug pour la console
    print(f"Page traitée : Gallica f{current_f} -> Archive _{archive_idx:04d}")

# =========================
# FINALISATION
# =========================
# Nettoyage namespaces XInclude
for el in root.xpath('//*[local-name()="include"]'):
    el.tag = "{http://www.w3.org/2001/XInclude}include"

tree.write("output.xml", encoding="UTF-8", xml_declaration=True, pretty_print=True)

# INHA

In [35]:
from lxml import etree
import re

# =========================
# PARAMÈTRES
# =========================
UUID = "282916be-b5f7-48c3-8171-6ce324e71bac"
PREFIXE_TIF = "0630_doucet_12res382_000005"

DOSSIER = "Peinture"
NOM_FICHIER_XML = "Piles_AbregeViePeintres"

START_F = 5  # numéro à 6 chiffres : 000005

BASE_URL = f"https://bibliotheque-numerique.inha.fr/i/?IIIF=/{UUID[0:2]}/{UUID[2:4]}/{UUID[4:6]}/{UUID[6:8]}/{UUID}/iiif/{PREFIXE_TIF}_{{}}/full/full/0/default.jpg"

NSMAP = {
    None: "http://www.tei-c.org/ns/1.0",
    "xi": "http://www.w3.org/2001/XInclude"
}

# =========================
# PARSER (IMPORTANT)
# =========================
parser = etree.XMLParser(
    remove_blank_text=True,
    strip_cdata=False
)

tree = etree.parse(f"../../corpus/{DOSSIER}/{NOM_FICHIER_XML}.xml", parser)
root = tree.getroot()

# =========================
# EXTRACTION numéro 6 chiffres
# =========================
def extract_f_number(facs_value):
    match = re.search(r'_(\d{6})\.tif', facs_value)
    if match:
        return int(match.group(1))
    return None

# =========================
# TRAITEMENT DES <pb>
# =========================
current_f = START_F
stop = False

for pb in root.xpath('//tei:pb', namespaces={'tei': NSMAP[None]}):
    facs = pb.get('facs')

    if facs == "stop":
        pb.attrib.pop('facs')
        stop = True
        continue

    if stop:
        continue

    if facs:
        extracted = extract_f_number(facs)
        if extracted:
            current_f = extracted
    else:
        current_f += 1
        pb.set('facs', BASE_URL.format(f"{current_f:06d}.tif"))

# =========================
# FIX xi:include (éviter ns1)
# =========================
for el in root.xpath('//*[local-name()="include"]'):
    el.tag = "{http://www.w3.org/2001/XInclude}include"

# =========================
# ÉCRITURE SANS ESPACES EN TROP
# =========================
tree.write(
    "output.xml",
    encoding="UTF-8",
    xml_declaration=True,
    pretty_print=True
)

print("✔ XML propre, namespaces conservés, pas de ns1")

✔ XML propre, namespaces conservés, pas de ns1


INHA avec <gap>

In [45]:
from lxml import etree
import re

# =========================
# PARAMÈTRES
# =========================
UUID = "282916be-b5f7-48c3-8171-6ce324e71bac"
PREFIXE_TIF = "0630_doucet_12res382"

DOSSIER = "Peinture"
NOM_FICHIER_XML = "Piles_AbregeViePeintres"

START_F = 4  # numéro à 6 chiffres : 000005

BASE_URL = f"https://bibliotheque-numerique.inha.fr/i/?IIIF=/{UUID[0:2]}/{UUID[2:4]}/{UUID[4:6]}/{UUID[6:8]}/{UUID}/iiif/{PREFIXE_TIF}_{{}}/full/full/0/default.jpg"

NSMAP = {
    None: "http://www.tei-c.org/ns/1.0",
    "xi": "http://www.w3.org/2001/XInclude"
}
TEI_NS = NSMAP[None]

# =========================
# PARSER (IMPORTANT)
# =========================
parser = etree.XMLParser(
    remove_blank_text=True,
    strip_cdata=False
)

tree = etree.parse(f"../../corpus/{DOSSIER}/{NOM_FICHIER_XML}.xml", parser)
root = tree.getroot()

# =========================
# EXTRACTION numéro 6 chiffres
# =========================
def extract_f_number(facs_value):
    match = re.search(r'_(\d{6})\.tif', facs_value)
    if match:
        return int(match.group(1))
    return None

# =========================
# TRAITEMENT DES <pb> ET DES <gap>
# =========================
# On parcourt pb ET gap ensemble, dans l'ordre du document (lxml conserve
# l'ordre du document avec l'opérateur "|"), pour que les <gap quantity="N">
# décalent bien le compteur "current_f" avant le prochain <pb>.
current_f = START_F
stop = False
after_gap = False  # True juste après un <gap> traité, pour forcer le recalcul du pb suivant

elements = root.xpath('//tei:pb | //tei:gap', namespaces={'tei': TEI_NS})

for el in elements:
    tag = etree.QName(el).localname

    # ---- Cas <gap> : décale le compteur de vues sans poser de facs ----
    if tag == 'gap':
        if stop:
            continue

        quantity = el.get('quantity')
        if quantity and quantity.isdigit():
            current_f += int(quantity)
            after_gap = True  # le prochain <pb> devra être recalculé, même s'il a déjà un facs
        else:
            print(f"  [!] <gap> sans @quantity exploitable (attrs={dict(el.attrib)}), ignoré")
        continue

    # ---- Cas <pb> : logique d'origine, avec priorité au <gap> qui précède ----
    facs = el.get('facs')

    if facs == "stop":
        el.attrib.pop('facs')
        stop = True
        after_gap = False
        continue

    if stop:
        continue

    if after_gap:
        # Un <gap> vient de décaler le compteur de "quantity" vues sautées.
        # Ce <pb> reste lui-même une page normale : on lui applique le même +1
        # qu'à n'importe quel pb sans facs, et on écrase tout facs déjà présent
        # (forcément calculé sans tenir compte de ce gap).
        ancienne_valeur = facs
        current_f += 1
        el.set('facs', BASE_URL.format(f"{current_f:06d}.tif"))
        if ancienne_valeur and ancienne_valeur != el.get('facs'):
            print(f"  [i] facs corrigé après <gap> : {ancienne_valeur} → {el.get('facs')}")
        after_gap = False
    elif facs:
        extracted = extract_f_number(facs)
        if extracted:
            current_f = extracted
    else:
        current_f += 1
        el.set('facs', BASE_URL.format(f"{current_f:06d}.tif"))

# =========================
# FIX xi:include (éviter ns1)
# =========================
for el in root.xpath('//*[local-name()="include"]'):
    el.tag = "{http://www.w3.org/2001/XInclude}include"

# =========================
# ÉCRITURE SANS ESPACES EN TROP
# =========================
tree.write(
    "output.xml",
    encoding="UTF-8",
    xml_declaration=True,
    pretty_print=True
)

print("✔ XML propre, namespaces conservés, pas de ns1, <gap quantity> pris en compte")

✔ XML propre, namespaces conservés, pas de ns1, <gap quantity> pris en compte
